In [ ]:
# Cell 1 — Install missing Colab packages (always first, no exceptions)
!pip install -r requirements-colab.txt -q

In [ ]:
# Cell 2 — Mount Google Drive
# Dataset at: MyDrive/aerial_detection/classification_dataset/
# Upload once — survives session restarts without re-uploading.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Clone repo and add to sys.path
# Replace GITHUB_URL with your actual repo URL before running.
import subprocess, sys

GITHUB_URL  = "https://github.com/YOUR_USERNAME/aerial-detection-production.git"
REPO_DIR    = "/content/aerial-detection-production"

result = subprocess.run(
    ["git", "clone", GITHUB_URL, REPO_DIR],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Add repo root so training.* imports resolve
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"sys.path[0]: {sys.path[0]}")

In [ ]:
# Cell 4 — Imports + Seeds
# stdlib
import json
import random
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

# third-party
import matplotlib.pyplot as plt
import numpy as np
import mlflow
import mlflow.keras
import tensorflow as tf

# internal (from cloned repo)
from training.data_loader import build_generators
from training.custom_cnn  import build_custom_cnn, compile_model, get_callbacks
from training.evaluate    import evaluate_model

# Seeds — must match RANDOM_STATE used throughout the project
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# File-based MLflow tracking — works on Colab without a tracking server
# Note: file-based MLflow does NOT support model registry (Staging/Production stages)
# Registry promotion is deferred to Section 8 (Docker MLflow with PostgreSQL backend)
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("aerial_custom_cnn")

print(f"TensorFlow: {tf.__version__}")
print(f"MLflow:     {mlflow.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Cell 5 — Config (all constants here, never inline)
IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 32
EPOCHS_CNN   = 50
PATIENCE     = 7
LEARNING_RATE = 1e-3
DROPOUT_RATE  = 0.5

DRIVE_ROOT   = Path("/content/drive/MyDrive/aerial_detection")
DATASET_DIR  = DRIVE_ROOT / "classification_dataset"
EDA_STATS    = Path(REPO_DIR) / "data" / "eda" / "eda_stats.json"
MODELS_DIR   = Path(REPO_DIR) / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Where trained artefacts are saved to Drive for persistence across sessions
DRIVE_MODELS = DRIVE_ROOT / "models"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH  = str(MODELS_DIR / "custom_cnn_best.h5")
FINAL_MODEL_PATH = str(MODELS_DIR / "custom_cnn_final.h5")

assert DATASET_DIR.exists(), f"Dataset not found at {DATASET_DIR} — check Drive mount"
assert EDA_STATS.exists(),   f"eda_stats.json not found at {EDA_STATS} — run EDA notebook first"

print(f"Dataset:    {DATASET_DIR}")
print(f"EDA stats:  {EDA_STATS}")
print(f"Models dir: {MODELS_DIR}")

In [ ]:
# Cell 6 — Load eda_stats.json (single source of truth for aug config + class_weight)
with open(EDA_STATS) as f:
    eda_stats = json.load(f)

aug_cfg      = eda_stats["augmentation_config"]
class_weight = {int(k): v for k, v in eda_stats["class_weight"].items()}

print("Augmentation config (from eda_stats.json):")
for k, v in aug_cfg.items():
    print(f"  {k}: {v}")

print(f"\nClass weight (from eda_stats.json): {class_weight}")
print(f"  bird=0 weight:  {class_weight[0]:.4f}  (majority class)")
print(f"  drone=1 weight: {class_weight[1]:.4f}  (minority class — drone misses penalised more)")

In [ ]:
# Cell 7 — Build data generators
# Reads augmentation config and class_weight from eda_stats.json — not hardcoded here.
# test_gen has shuffle=False — required for evaluate_model() to align predictions with labels.
train_gen, valid_gen, test_gen, class_weight_loaded, class_indices = build_generators(
    dataset_dir=DATASET_DIR,
    eda_stats_path=EDA_STATS,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
)

print(f"\nClass indices: {class_indices}")
print(f"Train batches: {len(train_gen)}  ({train_gen.n} images)")
print(f"Valid batches: {len(valid_gen)}  ({valid_gen.n} images)")
print(f"Test  batches: {len(test_gen)}   ({test_gen.n} images)")

In [ ]:
# Cell 8 — Build Custom CNN + print summary
# Architecture:
#   Block 1: Conv2D(32, relu) → BN → MaxPool(2×2)   224 → 112
#   Block 2: Conv2D(64, relu) → BN → MaxPool(2×2)   112 → 56
#   Block 3: Conv2D(128, relu) → BN → MaxPool(2×2)  56  → 28
#   Head:    GAP → Dense(128, relu) → Dropout(0.5) → Dense(1, sigmoid)
#
# GAP rationale: 28×28×128 feature map → Flatten = 12.8M params; GAP = 16,384 params.
# GAP is also translation-invariant — better for aerial images where object position varies.

model = build_custom_cnn(input_shape=(224, 224, 3), dropout_rate=DROPOUT_RATE)
model.summary()

total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\nTotal params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
# Cell 9 — Compile + Train with MLflow tracking
# class_weight passed to model.fit() — penalises drone misses more than false alarms
# (security system: FN is more costly than FP)

model = compile_model(model, learning_rate=LEARNING_RATE)
callbacks = get_callbacks(checkpoint_path=CHECKPOINT_PATH)

train_start = time.time()

with mlflow.start_run(run_name="custom_cnn_v1") as run:
    RUN_ID = run.info.run_id

    # Log all hyperparameters
    mlflow.log_params({
        "model":          "custom_cnn",
        "image_size":     str(IMAGE_SIZE),
        "batch_size":     BATCH_SIZE,
        "epochs_max":     EPOCHS_CNN,
        "learning_rate":  LEARNING_RATE,
        "dropout_rate":   DROPOUT_RATE,
        "patience":       PATIENCE,
        "class_weight_0": class_weight[0],
        "class_weight_1": class_weight[1],
        "optimizer":      "adam",
        "loss":           "binary_crossentropy",
        "normalization":  "divide_by_255",
    })
    mlflow.log_params({f"aug_{k}": v for k, v in aug_cfg.items()})

    history = model.fit(
        train_gen,
        validation_data=valid_gen,
        epochs=EPOCHS_CNN,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )

    train_time_s = time.time() - train_start

    # Log per-epoch metrics
    for epoch_idx, (loss, val_loss) in enumerate(
        zip(history.history["loss"], history.history["val_loss"])
    ):
        mlflow.log_metric("train_loss",  loss,    step=epoch_idx)
        mlflow.log_metric("val_loss",    val_loss, step=epoch_idx)
        if "accuracy" in history.history:
            mlflow.log_metric("train_acc", history.history["accuracy"][epoch_idx],     step=epoch_idx)
            mlflow.log_metric("val_acc",   history.history["val_accuracy"][epoch_idx], step=epoch_idx)

    epochs_run    = len(history.history["loss"])
    early_stopped = epochs_run < EPOCHS_CNN
    best_epoch    = int(np.argmin(history.history["val_loss"])) + 1
    final_lr      = float(history.history.get("lr", [LEARNING_RATE])[-1])

    mlflow.log_metrics({
        "epochs_run":     epochs_run,
        "best_epoch":     best_epoch,
        "final_lr":       final_lr,
        "train_time_min": round(train_time_s / 60, 2),
    })

print(f"\nTraining complete.")
print(f"  Epochs run:    {epochs_run} / {EPOCHS_CNN}")
print(f"  Early stopped: {early_stopped}")
print(f"  Best epoch:    {best_epoch}")
print(f"  Final LR:      {final_lr:.2e}")
print(f"  Train time:    {train_time_s/60:.1f} min")
print(f"  MLflow run ID: {RUN_ID}")

In [ ]:
# Cell 10 — Training Curves (loss + accuracy)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Custom CNN — Training History", fontsize=13)

epochs_range = range(1, epochs_run + 1)

# Loss
axes[0].plot(epochs_range, history.history["loss"],     label="Train loss",      color="steelblue")
axes[0].plot(epochs_range, history.history["val_loss"], label="Validation loss", color="darkorange")
axes[0].axvline(best_epoch, color="crimson", linestyle="--", label=f"Best epoch ({best_epoch})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
if "accuracy" in history.history:
    axes[1].plot(epochs_range, history.history["accuracy"],     label="Train accuracy",      color="steelblue")
    axes[1].plot(epochs_range, history.history["val_accuracy"], label="Validation accuracy", color="darkorange")
    axes[1].axvline(best_epoch, color="crimson", linestyle="--", label=f"Best epoch ({best_epoch})")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

plt.tight_layout()
curves_path = MODELS_DIR / "training_curves_custom_cnn.png"
plt.savefig(curves_path, dpi=100)
plt.show()
print(f"Saved: {curves_path}")

In [ ]:
# Cell 11 — Evaluate on test set
# evaluate_model() returns metrics at both threshold=0.5 and Youden-optimal threshold.
# Confusion matrix and ROC curve PNGs are saved to MODELS_DIR.
# EarlyStopping(restore_best_weights=True) ensures the model already holds best-epoch weights.

eval_metrics = evaluate_model(
    model=model,
    test_generator=test_gen,
    model_name="custom_cnn",
    output_dir=MODELS_DIR,
)

m_default = eval_metrics["metrics_at_default_threshold"]
m_opt     = eval_metrics["metrics_at_optimal_threshold"]
thresh    = eval_metrics["threshold_analysis"]

print("\n── Test metrics at threshold=0.5 (default) ──────────────────")
for k, v in m_default.items():
    print(f"  {k:<12}: {v}")

print(f"\n── Test metrics at Youden-optimal threshold ({thresh['optimal_threshold']:.3f}) ──")
for k, v in m_opt.items():
    print(f"  {k:<12}: {v}")

print(f"\n  Youden optimal — TPR: {thresh['optimal_tpr']:.4f}  FPR: {thresh['optimal_fpr']:.4f}")
print(f"  Eval time: {eval_metrics['eval_time_s']:.2f}s")

In [ ]:
# Cell 12 — Save metrics_cnn.json + model .h5 to Drive
# metrics_cnn.json uses default-threshold metrics in 'metrics' key (standardised schema
# for model comparison table in Section 4); optimal threshold is preserved separately.

# Compute test_loss from generator
test_gen.reset()
test_loss, *_ = model.evaluate(test_gen, verbose=0)

metrics_cnn = {
    "model_name":   "custom_cnn",
    "architecture": "3-block CNN + GAP + Dense(128) + sigmoid",
    "training": {
        "epochs_run":    epochs_run,
        "epochs_max":    EPOCHS_CNN,
        "early_stopped": early_stopped,
        "best_epoch":    best_epoch,
        "final_lr":      final_lr,
    },
    "metrics": {
        "test_accuracy":  m_default["accuracy"],
        "test_precision": m_default["precision"],
        "test_recall":    m_default["recall"],
        "test_f1":        m_default["f1_score"],
        "test_auc":       m_default["auc"],
        "test_loss":      round(float(test_loss), 4),
    },
    "optimal_threshold": {
        "threshold":  eval_metrics["threshold_analysis"]["optimal_threshold"],
        "accuracy":   m_opt["accuracy"],
        "precision":  m_opt["precision"],
        "recall":     m_opt["recall"],
        "f1_score":   m_opt["f1_score"],
        "criterion":  "youden_j",
    },
    "artifacts": {
        "model_file":        FINAL_MODEL_PATH,
        "confusion_matrix":  str(MODELS_DIR / "confusion_matrix_custom_cnn.png"),
        "roc_curve":         str(MODELS_DIR / "roc_curve_custom_cnn.png"),
        "training_curves":   str(MODELS_DIR / "training_curves_custom_cnn.png"),
        "mlflow_run_id":     RUN_ID,
        "mlflow_experiment": "aerial_custom_cnn",
    },
    "training_time_minutes": round(train_time_s / 60, 2),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

metrics_path = MODELS_DIR / "metrics_cnn.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_cnn, f, indent=2)

# Save full model (architecture + weights) — not just weights
model.save(FINAL_MODEL_PATH)

# Copy artefacts to Drive for persistence
for src in [
    FINAL_MODEL_PATH,
    str(MODELS_DIR / "custom_cnn_best.h5"),
    str(metrics_path),
    str(MODELS_DIR / "confusion_matrix_custom_cnn.png"),
    str(MODELS_DIR / "roc_curve_custom_cnn.png"),
    str(MODELS_DIR / "training_curves_custom_cnn.png"),
]:
    if Path(src).exists():
        shutil.copy2(src, DRIVE_MODELS)
        print(f"Copied to Drive: {Path(src).name}")

print(f"\nmetrics_cnn.json:")
print(f"  test_accuracy:  {metrics_cnn['metrics']['test_accuracy']}")
print(f"  test_f1:        {metrics_cnn['metrics']['test_f1']}")
print(f"  test_auc:       {metrics_cnn['metrics']['test_auc']}")
print(f"  optimal_thr:    {metrics_cnn['optimal_threshold']['threshold']}")

In [ ]:
# Cell 13 — MLflow: log artifacts + zip mlruns to Drive
#
# NOTE: file-based MLflow (file:./mlruns) does NOT support the Model Registry.
# mlflow.register_model() and client.transition_model_version_stage() both require
# a database-backed tracking server (PostgreSQL or SQLite).
# Registry promotion is deferred to Section 8 (Docker MLflow + PostgreSQL backend).
# production_model.txt written by Section 4's build_comparison_report() is used
# by FastAPI to identify which model to load at startup.

with mlflow.start_run(run_id=RUN_ID):
    # Log scalar metrics from evaluation
    mlflow.log_metrics({
        "test_accuracy":     metrics_cnn["metrics"]["test_accuracy"],
        "test_precision":    metrics_cnn["metrics"]["test_precision"],
        "test_recall":       metrics_cnn["metrics"]["test_recall"],
        "test_f1":           metrics_cnn["metrics"]["test_f1"],
        "test_auc":          metrics_cnn["metrics"]["test_auc"],
        "test_loss":         metrics_cnn["metrics"]["test_loss"],
        "optimal_threshold": metrics_cnn["optimal_threshold"]["threshold"],
        "optimal_f1":        metrics_cnn["optimal_threshold"]["f1_score"],
    })

    # Log artifacts — model file + evaluation PNGs + metrics JSON
    mlflow.log_artifact(FINAL_MODEL_PATH)
    mlflow.log_artifact(str(metrics_path))
    mlflow.log_artifact(str(MODELS_DIR / "confusion_matrix_custom_cnn.png"))
    mlflow.log_artifact(str(MODELS_DIR / "roc_curve_custom_cnn.png"))
    mlflow.log_artifact(str(MODELS_DIR / "training_curves_custom_cnn.png"))

print(f"MLflow run {RUN_ID} updated with test metrics and artifacts.")
print("Registry promotion deferred to Section 8 (Docker MLflow + PostgreSQL).")

# Zip mlruns to Drive — file-based tracking is ephemeral on Colab; Drive survives restarts
mlruns_zip = DRIVE_ROOT / "mlruns_custom_cnn"
shutil.make_archive(str(mlruns_zip), "zip", root_dir=".", base_dir="mlruns")
print(f"Zipped mlruns → {mlruns_zip}.zip")

print("\n── Section 3 Complete ──────────────────────────────────────")
print(f"  model:    custom_cnn_final.h5  (Drive: {DRIVE_MODELS})")
print(f"  metrics:  metrics_cnn.json     (Drive + MODELS_DIR)")
print(f"  mlflow:   run_id={RUN_ID}")
print("  registry: deferred to Section 8")